In [ ]:
!pip install langchain_core
!pip install langchain
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.2 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

gemini_api = userdata.get('gemini_api')

model = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    api_key = gemini_api,
    temperature = 0.8
)

**Runnables are a way to run chains, data retrievers, vectors.. in a straight way**

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

template1 = PromptTemplate(
    template = 'Tell me a joke about {topic}',
    input_variables = ['topic'],
)

template2 = PromptTemplate(
    template = 'give me a summary about {joke}',
    input_variables = ['joke']
)

In [ ]:
from langchain_core.runnables import RunnableSequence

runnable_sequence = RunnableSequence(template1, model, str_parser, template2, model, str_parser)

In [ ]:
print(runnable_sequence.invoke('cats'))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 16.429893763s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

*Runnable Parallel*

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_template1 = PromptTemplate(
    template = 'Tell me a tweet about {topic} from X (twitter)',
    input_variables = ['topic'],
)

parallel_template2 = PromptTemplate(
    template = 'Tell me a post about {topic} from linkedin',
    input_variables = ['topic']
)

runnable_parallel = RunnableParallel(
    tweet = RunnableSequence(parallel_template1, model, str_parser),
    post = RunnableSequence(parallel_template2, model, str_parser)
)

runnable_parallel.invoke('AI')

**The input  (joke) got missed to the second chain in the sequence chain**

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

joke_gen_chain = RunnableSequence(template1, model, str_parser)

summary_joke_chain = RunnableParallel(
    joke = RunnablePassthrough(),
    summary = RunnableSequence(template2, model, str_parser)
)

complete_chain = RunnableSequence(joke_gen_chain, summary_joke_chain)

print(complete_chain.invoke('cats'))



In [ ]:
#we will get the topic of the joke in the output

joke_gen_chain2 = RunnableParallel(
    topic = RunnablePassthrough(),
    joke = RunnableSequence(template1, model, str_parser)
)

summary_joke_chain2 = RunnableParallel(
    topic = lambda x : x['topic'],
    joke = lambda x : x['joke'],
    summary = (lambda x : x['joke']) | RunnableSequence(template2, model, str_parser)
)

complete_chain2 = RunnableSequence(joke_gen_chain2, summary_joke_chain2)

complete_chain2.invoke('cats')

In [ ]:
#cleaner way to assign

summary_joke_chain3 = RunnablePassthrough.assign(
    summary = (lambda x: x['joke']) | RunnableSequence(template2, model, str_parser)
)

complete_chain3 = RunnableSequence(joke_gen_chain2, summary_joke_chain3)

complete_chain3.invoke('cats')

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

def word_count(text):
  return len(text.split())

template1 = PromptTemplate(
    template = 'give me a summary about {topic}',
    input_variables = ['topic'],
)

template2 = PromptTemplate(
    template = 'Give me summary of the text\n {summary}',
    input_variables = ['summary']
)

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableBranch, RunnableSequence

# 1. Extract the string value of the topic immediately so it doesn't nest
summary_chain = RunnableParallel(
    topic=lambda x: x['topic'],
    summary=RunnableSequence(template1, model, str_parser)
)

# 2. Update the branch fallback to cleanly return just the 'summary' string
branch_chain = RunnableBranch(
    (lambda x: word_count(x['summary']) > 50, RunnableSequence(template2, model, str_parser)),
    lambda x: x['summary']
)

# 3. Cleanly assign the branch output to 'final'
complete_chain = summary_chain.assign(
    final=branch_chain
)

# Execute the chain
result = complete_chain.invoke({'topic': 'Russia vs Ukraine war'})
print(result)